# **EEG Preprocessing in Python MNE**


# Basic pipeline
Below we go through a basic preprocessing pipeline using a sample dataset from MNE.

## Import the few packages we'll need

In [1]:
import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import mne
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
import numpy as np


In [2]:
from mne.preprocessing import ICA
from mne_icalabel import label_components
from sklearn.neighbors import NearestCentroid
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
%matplotlib inline

## Load data

In [4]:

raw = mne.io.read_raw_eeglab("sub-01/ses-S1/eeg/zeroBACK.set", preload=True)
print(raw.info)
print("Channel count:", len(raw.ch_names))
print(raw.ch_names)
print("Sampling rate:", raw.info["sfreq"])

Reading /Users/gulkarabas/Documents/bci-exp/sub-01/ses-S1/eeg/zeroBACK.fdt
Reading 0 ... 198287  =      0.000 ...   396.574 secs...
<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, Fz, F3, F7, FT9, FC5, FC1, C3, T7, ECG1, CP5, CP1, Pz, P3, ...
 chs: 62 EEG, 1 ECG
 custom_ref_applied: False
 dig: 65 items (3 Cardinal, 62 EEG)
 highpass: 0.0 Hz
 lowpass: 250.0 Hz
 meas_date: unspecified
 nchan: 63
 projs: []
 sfreq: 500.0 Hz
>
Channel count: 63
['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'ECG1', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'FCz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2']
Sampling rate: 500.0


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1146887538.py:1: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab("sub-01/ses-S1/eeg/zeroBACK.set", preload=True)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1146887538.py:1: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab("sub-01/ses-S1/eeg/zeroBACK.set", preload=True)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1146887538.py:1: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab("sub-01/ses-S1/eeg/zeroBACK.set", preload=True)


In [5]:
if "ECG" in raw.ch_names:
    raw.drop_channels(["ECG"])
print("EEG channels remaining:", len(raw.ch_names))

EEG channels remaining: 63


In [6]:
raw.filter(l_freq=1.0, h_freq=45.0)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1651 samples (3.302 s)



<RawEEGLAB | zeroBACK.fdt, 63 x 198288 (396.6 s), ~95.4 MiB, data loaded>

In [7]:
print(raw.annotations)
print(set(raw.annotations.description))

<Annotations | 203 segments: 601 (1), 6011 (6), 6012 (3), 6021 (96), 6022 ...>
{np.str_('boundary'), np.str_('6031'), np.str_('6032'), np.str_('6011'), np.str_('601'), np.str_('6022'), np.str_('6012'), np.str_('6021')}


In [ ]:

events, event_id = mne.events_from_annotations(raw)
print(event_id)


unique, counts = np.unique(events[:, 2], return_counts=True)
for code, n in zip(unique, counts):
    name = [k for k, v in event_id.items() if v == code][0]
    print(name, n)

Used Annotations descriptions: [np.str_('601'), np.str_('6011'), np.str_('6012'), np.str_('6021'), np.str_('6022'), np.str_('6031'), np.str_('6032'), np.str_('boundary')]
{np.str_('601'): 1, np.str_('6011'): 2, np.str_('6012'): 3, np.str_('6021'): 4, np.str_('6022'): 5, np.str_('6031'): 6, np.str_('6032'): 7, np.str_('boundary'): 8}
601 1
6011 6
6012 3
6021 96
6022 48
6031 1
6032 47
boundary 1


In [9]:
stim_ids = {"6021": event_id["6021"], "6022": event_id["6022"]}

tmin, tmax = 0.0, 2.0
epochs = mne.Epochs(
    raw, events, event_id=stim_ids,
    tmin=tmin, tmax=tmax, baseline=None, preload=True
)
print(epochs)

Not setting metadata
144 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 144 events and 1001 original time points ...
0 bad epochs dropped
<Epochs | 144 events (all good), 0 – 2 s (baseline off), ~69.4 MiB, data loaded,
 '6021': 96
 '6022': 48>


In [10]:
bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}

spectrum = epochs.compute_psd(method="welch", fmin=4.0, fmax=30.0, verbose=False)
psds, freqs = spectrum.get_data(return_freqs=True)   # shape: (n_epochs, n_channels, n_freqs)



In [11]:
band_powers = []
for name, (low, high) in bands.items():
    mask = (freqs >= low) & (freqs < high)
    bp = psds[:, :, mask].mean(axis=2)               
    band_powers.append(bp)

X = np.concatenate(band_powers, axis=1)              
X = np.log(X)
print("Feature matrix shape:", X.shape)

Feature matrix shape: (144, 186)


## loop

In [12]:
STIM_CODES = {
    0: ["6021", "6022"],   # zeroBACK nontarget target
    1: ["6221", "6222"],   # twoBACK  nontarget target
}

def extract_features(filepath, label):
    raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
    raw.pick("eeg")
    raw.filter(l_freq=1.0, h_freq=45.0, verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)
    stim_ids = {name: event_id[name] for name in STIM_CODES[label] if name in event_id}
    if not stim_ids:
        raise ValueError(f"No stimulus markers {STIM_CODES[label]} found in {filepath}")

    epochs = mne.Epochs(raw, events, event_id=stim_ids, tmin=0.0, tmax=2.0,
                        baseline=None, preload=True, verbose=False)

    spectrum = epochs.compute_psd(method="welch", fmin=4.0, fmax=30.0, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)

    bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}
    band_powers = []
    for low, high in bands.values():
        mask = (freqs >= low) & (freqs < high)
        band_powers.append(psds[:, :, mask].mean(axis=2))

    X = np.log(np.concatenate(band_powers, axis=1))
    y = np.full(len(X), label)
    return X, y

## save

In [13]:
files = {
    ("S1", 0): "sub-01/ses-S1/eeg/zeroBACK.set",
    ("S1", 1): "sub-01/ses-S1/eeg/twoBACK.set",
    ("S2", 0): "sub-01/ses-S2/eeg/zeroBACK.set",
    ("S2", 1): "sub-01/ses-S2/eeg/twoBACK.set",
    ("S3", 0): "sub-01/ses-S3/eeg/zeroBACK.set",
    ("S3", 1): "sub-01/ses-S3/eeg/twoBACK.set",
}

parts = {}
for (session, label), path in files.items():
    X, y = extract_features(path, label)
    parts.setdefault(session, []).append((X, y))
    print(f"{session} label={label}: {X.shape[0]} epochs, {X.shape[1]} features")

/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S1 label=0: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S1 label=1: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


...

S2 label=1: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S3 label=0: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S3 label=1: 144 epochs, 186 features


In [14]:
parts = {}
for (session, label), path in files.items():
    X, y = extract_features(path, label)
    parts.setdefault(session, []).append((X, y))
    print(f"{session} label={label}: {X.shape[0]} epochs, {X.shape[1]} features")

session_data = {}
for session, items in parts.items():
    Xs = np.concatenate([X for X, _ in items], axis=0)
    ys = np.concatenate([y for _, y in items], axis=0)
    session_data[session] = (Xs, ys)
    print(session, Xs.shape, "label counts:", np.bincount(ys))

/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S1 label=0: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S1 label=1: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


...

S2 label=1: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S3 label=0: 144 epochs, 186 features


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/985512585.py:7: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


S3 label=1: 144 epochs, 186 features
S1 (288, 186) label counts: [144 144]
S2 (288, 186) label counts: [144 144]
S3 (288, 186) label counts: [144 144]


In [15]:
session_data = {}
for session, items in parts.items():
    Xs = np.concatenate([X for X, _ in items], axis=0)
    ys = np.concatenate([y for _, y in items], axis=0)
    session_data[session] = (Xs, ys)
    print(session, Xs.shape, "label counts:", np.bincount(ys))

S1 (288, 186) label counts: [144 144]
S2 (288, 186) label counts: [144 144]
S3 (288, 186) label counts: [144 144]


### Fit the scaler on day 1 ONLY 


In [16]:
X_train, y_train = session_data["S1"]
scaler = StandardScaler().fit(X_train)

Xs = {s: scaler.transform(session_data[s][0]) for s in ["S1", "S2", "S3"]}
ys = {s: session_data[s][1] for s in ["S1", "S2", "S3"]}

#### A centroid is the mean feature vector of each class on day 1


In [17]:
classes = [0, 1]
centroids = {c: Xs["S1"][ys["S1"] == c].mean(axis=0) for c in classes}

In [18]:
def predict(x, centroids):
    dists = {c: np.linalg.norm(x - centroid) for c, centroid in centroids.items()}
    return min(dists, key=dists.get)

In [19]:
def evaluate_static(centroids, X, y):
    preds = [predict(x, centroids) for x in X]
    return balanced_accuracy_score(y, preds)

for s in ["S2", "S3"]:
    acc = evaluate_static(centroids, Xs[s], ys[s])
    print(f"Static on {s}: balanced accuracy = {acc:.3f}")

Static on S2: balanced accuracy = 0.531
Static on S3: balanced accuracy = 0.667


### online

In [ ]:
def evaluate_online(init_centroids, stream_X, stream_y, alpha=0.10):
    centroids = {c: v.copy() for c, v in init_centroids.items()}
    preds = []
    for x, true_label in zip(stream_X, stream_y):
        preds.append(predict(x, centroids))                                       # predict first
        centroids[true_label] = (1 - alpha) * centroids[true_label] + alpha * x    # then update
    return balanced_accuracy_score(stream_y, preds)

def persistence_baseline(stream_y):
    # Predict the previous label 
    preds = np.concatenate([[stream_y[0]], stream_y[:-1]])
    return balanced_accuracy_score(stream_y, preds)

#  day 2 then day 3 one 0 back block followed by one 2 back block
stream_X = np.concatenate([Xs["S2"], Xs["S3"]], axis=0)
stream_y = np.concatenate([ys["S2"], ys["S3"]], axis=0)

static_acc  = balanced_accuracy_score(stream_y, [predict(x, centroids) for x in stream_X])
online_acc  = evaluate_online(centroids, stream_X, stream_y, alpha=0.10)
persist_acc = persistence_baseline(stream_y)

print(f"Static      (sequential): {static_acc:.3f}")
print(f"Online      (sequential): {online_acc:.3f}")
print(f"Persistence (sequential): {persist_acc:.3f}")

Static      (sequential): 0.599
Online      (sequential): 0.830
Persistence (sequential): 0.995


In [21]:
rng = np.random.default_rng(42)
order = rng.permutation(len(stream_y))
stream_X_il = stream_X[order]
stream_y_il = stream_y[order]

static_il  = balanced_accuracy_score(stream_y_il, [predict(x, centroids) for x in stream_X_il])
online_il  = evaluate_online(centroids, stream_X_il, stream_y_il, alpha=0.10)
persist_il = persistence_baseline(stream_y_il)

print(f"Static      (interleave): {static_il:.3f}")
print(f"Online      (interleave): {online_il:.3f}")
print(f"Persistence (interleave): {persist_il:.3f}")

Static      (interleave): 0.599
Online      (interleave): 0.616
Persistence (interleave): 0.523


In [22]:
print(session_data["S1"][0].shape)   

(288, 186)


### ICA

Note on the pipeline update. Up to this point the pipeline used no average reference. To keep the noICA vs. ICA comparison fair, an average reference is applied to both arms from here on. The cells above are the earlier. 

In [23]:
def extract_features(filepath, label, apply_ica=False):
    raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
    raw.pick("eeg")
    raw.filter(l_freq=1.0, h_freq=45.0, verbose=False)
    raw.set_eeg_reference("average", verbose=False)   # same reference for both arms

    if apply_ica:
        ica = ICA(n_components=0.95, method="infomax",
                  fit_params=dict(extended=True), random_state=42, verbose=False)
        ica.fit(raw, verbose=False)
        result = label_components(raw, ica, method="iclabel")
        exclude = [i for i, (lab, prob) in enumerate(zip(result["labels"], result["y_pred_proba"]))
                   if lab not in ["brain", "other"] and prob >= 0.80]
        print(f"{filepath}: excluding {len(exclude)} -> {[result['labels'][i] for i in exclude]}")
        ica.apply(raw, exclude=exclude, verbose=False)

    events, event_id = mne.events_from_annotations(raw, verbose=False)
    stim_ids = {name: event_id[name] for name in STIM_CODES[label] if name in event_id}
    if not stim_ids:
        raise ValueError(f"No stimulus markers {STIM_CODES[label]} found in {filepath}")

    epochs = mne.Epochs(raw, events, event_id=stim_ids, tmin=0.0, tmax=2.0,
                        baseline=None, preload=True, verbose=False)
    spectrum = epochs.compute_psd(method="welch", fmin=4.0, fmax=30.0, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)

    bands = {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}
    band_powers = []
    for low, high in bands.values():
        mask = (freqs >= low) & (freqs < high)
        band_powers.append(psds[:, :, mask].mean(axis=2))

    X = np.log(np.concatenate(band_powers, axis=1))
    y = np.full(len(X), label)
    return X, y

In [24]:
def build_session_data(apply_ica):
    parts = {}
    for (session, label), path in files.items():
        X, y = extract_features(path, label, apply_ica=apply_ica)
        parts.setdefault(session, []).append((X, y))
    return {s: (np.concatenate([X for X, _ in v]), np.concatenate([y for _, y in v]))
            for s, v in parts.items()}

def run_comparison(session_data):
    scaler = StandardScaler().fit(session_data["S1"][0])
    Xs = {s: scaler.transform(session_data[s][0]) for s in ["S1", "S2", "S3"]}
    ys = {s: session_data[s][1] for s in ["S1", "S2", "S3"]}
    centroids = {c: Xs["S1"][ys["S1"] == c].mean(axis=0) for c in [0, 1]}

    stream_X = np.concatenate([Xs["S2"], Xs["S3"]])
    stream_y = np.concatenate([ys["S2"], ys["S3"]])

    rng = np.random.default_rng(42)
    order = rng.permutation(len(stream_y))
    sx, sy = stream_X[order], stream_y[order]   

    return {
        "static":      balanced_accuracy_score(sy, [predict(x, centroids) for x in sx]),
        "online":      evaluate_online(centroids, sx, sy, alpha=0.10),
        "persistence": persistence_baseline(sy),
    }

In [25]:
results_noica = run_comparison(build_session_data(apply_ica=False))
results_ica   = run_comparison(build_session_data(apply_ica=True))

print("no_ica:", {k: round(v, 3) for k, v in results_noica.items()})
print("ica:   ", {k: round(v, 3) for k, v in results_ica.items()})

/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S1/eeg/zeroBACK.set: excluding 2 -> ['eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S1/eeg/twoBACK.set: excluding 3 -> ['eye blink', 'eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

...

sub-01/ses-S2/eeg/twoBACK.set: excluding 2 -> ['eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S3/eeg/zeroBACK.set: excluding 2 -> ['eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S3/eeg/twoBACK.set: excluding 1 -> ['eye blink']
no_ica: {'static': 0.536, 'online': 0.601, 'persistence': 0.523}
ica:    {'static': 0.611, 'online': 0.708, 'persistence': 0.523}


In [26]:
data_noica = build_session_data(apply_ica=False)
data_ica   = build_session_data(apply_ica=True)

/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S1/eeg/zeroBACK.set: excluding 2 -> ['eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S1/eeg/twoBACK.set: excluding 3 -> ['eye blink', 'eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

...

sub-01/ses-S2/eeg/twoBACK.set: excluding 2 -> ['eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S3/eeg/zeroBACK.set: excluding 2 -> ['eye blink', 'eye blink']


/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipykernel_71962/1430677541.py:2: RuntimeWarning: Not setting position of 1 ecg channel found in montage:
['ECG1']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/var/folders/_c/pzmcpdh93jzf1fhchr8jxqt40000gn/T/ipyk

sub-01/ses-S3/eeg/twoBACK.set: excluding 1 -> ['eye blink']


In [27]:
def run_comparison(session_data, seeds=range(10), alpha=0.10):
    scaler = StandardScaler().fit(session_data["S1"][0])
    Xs = {s: scaler.transform(session_data[s][0]) for s in ["S1", "S2", "S3"]}
    ys = {s: session_data[s][1] for s in ["S1", "S2", "S3"]}
    centroids = {c: Xs["S1"][ys["S1"] == c].mean(axis=0) for c in [0, 1]}

    stream_X = np.concatenate([Xs["S2"], Xs["S3"]])
    stream_y = np.concatenate([ys["S2"], ys["S3"]])

    static = balanced_accuracy_score(stream_y, [predict(x, centroids) for x in stream_X])  

    online_scores, persist_scores = [], []
    for seed in seeds:
        order = np.random.default_rng(seed).permutation(len(stream_y))
        sx, sy = stream_X[order], stream_y[order]
        online_scores.append(evaluate_online(centroids, sx, sy, alpha=alpha))
        persist_scores.append(persistence_baseline(sy))

    return {
        "static": round(static, 3),
        "online_mean": round(np.mean(online_scores), 3),
        "online_std": round(np.std(online_scores), 3),
        "persistence_mean": round(np.mean(persist_scores), 3),
    }

print("no_ica:", run_comparison(data_noica))
print("ica:   ", run_comparison(data_ica))

no_ica: {'static': 0.536, 'online_mean': np.float64(0.628), 'online_std': np.float64(0.011), 'persistence_mean': np.float64(0.507)}
ica:    {'static': 0.611, 'online_mean': np.float64(0.721), 'online_std': np.float64(0.01), 'persistence_mean': np.float64(0.507)}


In [28]:
for alpha in [0.02, 0.05, 0.10, 0.20]:
    print(f"alpha={alpha}")
    print("  no_ica:", run_comparison(data_noica, alpha=alpha))
    print("  ica:   ", run_comparison(data_ica, alpha=alpha))

alpha=0.02
  no_ica: {'static': 0.536, 'online_mean': np.float64(0.663), 'online_std': np.float64(0.009), 'persistence_mean': np.float64(0.507)}
  ica:    {'static': 0.611, 'online_mean': np.float64(0.738), 'online_std': np.float64(0.01), 'persistence_mean': np.float64(0.507)}
alpha=0.05
  no_ica: {'static': 0.536, 'online_mean': np.float64(0.649), 'online_std': np.float64(0.009), 'persistence_mean': np.float64(0.507)}
  ica:    {'static': 0.611, 'online_mean': np.float64(0.737), 'online_std': np.float64(0.009), 'persistence_mean': np.float64(0.507)}
alpha=0.1
  no_ica: {'static': 0.536, 'online_mean': np.float64(0.628), 'online_std': np.float64(0.011), 'persistence_mean': np.float64(0.507)}
  ica:    {'static': 0.611, 'online_mean': np.float64(0.721), 'online_std': np.float64(0.01), 'persistence_mean': np.float64(0.507)}
alpha=0.2
  no_ica: {'static': 0.536, 'online_mean': np.float64(0.59), 'online_std': np.float64(0.013), 'persistence_mean': np.float64(0.507)}
  ica:    {'static': 0.

In [29]:

def within_session_accuracy(session_data, n_splits=5, seed=42):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    results = {}
    for s in ["S1", "S2", "S3"]:
        X, y = session_data[s]
        clf = make_pipeline(StandardScaler(), NearestCentroid())   
        scores = cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy")
        results[s] = round(scores.mean(), 3)
    return results

print("within-session (no_ica):", within_session_accuracy(data_noica))
print("within-session (ica):   ", within_session_accuracy(data_ica))

within-session (no_ica): {'S1': np.float64(0.573), 'S2': np.float64(0.729), 'S3': np.float64(0.733)}
within-session (ica):    {'S1': np.float64(0.702), 'S2': np.float64(0.878), 'S3': np.float64(0.799)}
